# CryptoStatArb — Research Notebook

**Statistical Arbitrage in Cryptocurrencies** (WSQ Course Project)

Goal: research profitable **momentum and/or reversal** strategies on crypto,
backtest unconstrained, apply realistic execution costs (Binance.US, Tier 1),
and evaluate returns / vol / Sharpe / max drawdown / alpha-beta.

Exchange: [Binance.US](https://www.binance.us/fees)

In [ ]:
# --- Core data analysis ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Binance.US API (python-binance) ---
from binance.client import Client
from binance.exceptions import BinanceAPIException

# --- Stats / research ---
import scipy.stats as stats
import statsmodels.api as sm

# --- Utilities ---
import os
import time
import datetime as dt
from pathlib import Path
from importlib.metadata import version

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')
sns.set_theme(style='whitegrid')
%matplotlib inline

print('python-binance', version('python-binance'))
print('pandas', pd.__version__, '| numpy', np.__version__)

## Fees & Transaction Costs

Source: <https://www.binance.us/fees> (Tier 1, volume TBD). Commission is tracked
against the live schedule; **bid/ask spread and market-impact costs are asset-dependent**
and set per symbol during research. See [`docs/fees.md`](docs/fees.md).

In [ ]:
# --- Binance.US commission (published standard rates, checked 2026-08-27) ---
FEES = {
    'maker_bps': 0.0,      # 0.00%  limit orders that ADD liquidity
    'taker_bps': 2.0,      # 0.02%  market orders that TAKE liquidity
    'bnb_discount': 0.05,  # 5% off maker & taker when fees paid in BNB
}

# Course-project baseline (from ClassProject brief) for cross-checking backtests:
#   market order = 7 bps commission + 13 bps assumed slippage = 20 bps all-in
#   limit order  = 7 bps commission only
PROJECT_COST_ASSUMPTION = {
    'commission_bps': 7.0,
    'slippage_bps': 13.0,
    'market_all_in_bps': 20.0,
    'limit_bps': 7.0,
}

# Asset-dependent costs (half-spread, impact) are filled in per symbol later, e.g.:
#   SYMBOL_TCOSTS = {'BTCUSDT': {'half_spread_bps': ..., 'impact_bps': ...}, ...}
SYMBOL_TCOSTS = {}

def bps(x):
    """basis points -> decimal (e.g. 20 -> 0.0020)."""
    return x / 1e4

FEES

## Data

Two paths: (1) load the week-3 lecture dataset already saved locally, or
(2) pull fresh OHLCV from Binance.US via `python-binance`. Public market data
(klines) needs **no API key**; keys are only required for account/trading endpoints
and must live in a gitignored `.env` — never commit them.

In [ ]:
# Option 1 — load the existing lecture dataset (Close/High/Low/Open/Volume x tickers)
DATA_DIR = Path('data')
px = pd.read_pickle(DATA_DIR / 'crypto_px.pk')
print(px.shape)
px.tail()

In [ ]:
# Option 2 — pull fresh public OHLCV from Binance.US (no key needed for market data)
# client = Client(tld='us')  # public endpoints only
#
# def get_klines(symbol, interval=Client.KLINE_INTERVAL_1DAY, start='1 Jan 2019'):
#     raw = client.get_historical_klines(symbol, interval, start)
#     cols = ['open_time','open','high','low','close','volume','close_time',
#             'qav','trades','tbbav','tbqav','ignore']
#     df = pd.DataFrame(raw, columns=cols)
#     df['date'] = pd.to_datetime(df['open_time'], unit='ms')
#     for c in ['open','high','low','close','volume']:
#         df[c] = df[c].astype(float)
#     return df.set_index('date')[['open','high','low','close','volume']]
#
# btc = get_klines('BTCUSDT')
# btc.tail()